# JetRacer calibration

**Intelligent Vehicles · JetRacer practical lab**  
**Hardware:** Waveshare JetRacer Pro, Jetson Nano 4 GB  
**Software:** JetPack 4.5.1, Python 3.6, JetRacer `ws/pro` branch  


Set the steering and throttle for one JetRacer. Test the values. Save them in a file. Load the file in another notebook.

At the end of this lab, you can:

1. Explain `steering`, `steering_gain`, `steering_offset`, `throttle`, and `throttle_gain`.
2. Find the straight steering position.
3. Find safe left and right steering commands.
4. Find the lowest useful throttle on the test surface.
5. Use a slider to control the steering.
6. Save and load a calibration file.

**Rule:** Calibrate each car. Do not copy the measured values from another car without a test.


## Find the JetRacer control file

The `NvidiaRacecar` class sends steering and throttle commands to the motor controller.

The source file is normally in:

`/home/jetson/jetracer/jetracer/nvidia_racecar.py`

The installed copy can be in a Python `.egg` directory. Read the installed location below. **Do not edit the driver for this lab.**


In [2]:
import inspect
from jetracer.nvidia_racecar import NvidiaRacecar

print("Installed control file:")
print(inspect.getfile(NvidiaRacecar))

print("\nDefault values:")
for name in ("steering_gain", "steering_offset", "throttle_gain"):
    print(name, "=", NvidiaRacecar.class_traits()[name].default_value)


Installed control file:
/usr/local/lib/python3.6/dist-packages/jetracer-0.0.0-py3.6.egg/jetracer/nvidia_racecar.py

Default values:
steering_gain = -0.65
steering_offset = 0
throttle_gain = 0.8


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,



| Name | Meaning | Example |
|---|---|---|
| `car.steering` | The requested turn | `0` = straight |
| `car.steering_gain` | How strongly the servo responds | `-0.65` |
| `car.steering_offset` | A correction for the center | `0.07` |
| `car.throttle` | The requested motor command | `0` = stop |
| `car.throttle_gain` | Scales the motor command | `0.8` |

The control file uses these calculations:

`servo_command = car.steering * car.steering_gain + car.steering_offset`

`motor_target = car.throttle * car.throttle_gain`

For example, a throttle of `0.12` and a gain of `0.8` give a motor target of `0.096`. This is **not** a speed in m/s or a power value in watts.

In this JetRacer branch, the motor also has a soft-start function. The soft-start function can change the motor command in steps.

The PWM controller normally runs at **60 Hz** in the Waveshare `ws/pro` class. This is the control-signal frequency. It is not the motor speed. Do not change it during this lab.

**Check:** What happens to the servo movement if you change the magnitude of `steering_gain` from `0.5` to `0.9`? What happens to the motor target if you reduce `throttle_gain`?


In [3]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()
car.throttle = 0.0

print("Steering gain:", car.steering_gain)
print("Steering offset:", car.steering_offset)
print("Throttle gain:", car.throttle_gain)

Steering gain: -0.65
Steering offset: 0.0
Throttle gain: 0.8


In [15]:
def set_center_offset(value):
    car.throttle = 0.0
    car.steering_offset = float(value)
    car.steering = 0.001
    car.steering = 0.0
    print("Steering offset:", car.steering_offset)


set_center_offset(0.05)

Steering offset: 0.05


In [22]:
def test_throttle(value, duration=1.0):
    value = float(value)
    if not 0.0 <= value <= 0.16:
        raise ValueError("Initial test range is 0.00 to 0.16")

    try:
        car.throttle = value
        print("Requested throttle:", value)
        print("Motor target:", value * car.throttle_gain)
        time.sleep(duration)
    finally:
        car.throttle = 0.0
        print("Stop command sent")


# Test one value at a time. Keep the wheels raised.
test_throttle(0.13)


Requested throttle: 0.13
Motor target: 0.10400000000000001
Stop command sent


In [25]:
car.throttle_gain = 0.8

In [34]:
import ipywidgets as widgets
from IPython.display import display

WIDGET_LIMIT = 0.90

steering_slider = widgets.FloatSlider(
    value=0.0,
    min=-WIDGET_LIMIT,
    max=WIDGET_LIMIT,
    step=0.01,
    description="Steering",
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

center_button = widgets.Button(description="Center wheels")
stop_button = widgets.Button(description="Stop test", button_style="danger")
steering_label = widgets.Label(value="Steering command: 0.00")


def slider_changed(change):
    value = float(change["new"])
    car.steering = value
    steering_label.value = "Steering command: {:.2f}".format(value)


def center_clicked(button):
    steering_slider.value = 0.0
    car.steering = 0.0


def stop_clicked(button):
    steering_slider.unobserve(slider_changed, names="value")
    steering_slider.disabled = True
    car.throttle = 0.0
    car.steering = 0.0
    steering_label.value = "Test stopped"


steering_slider.observe(slider_changed, names="value")
center_button.on_click(center_clicked)
stop_button.on_click(stop_clicked)

display(widgets.VBox([
    steering_slider,
    steering_label,
    widgets.HBox([center_button, stop_button])
]))
